<a href="https://colab.research.google.com/github/marifinilham-ds/east-java-positive-deviance-few/blob/main/notebooks/01_earth_engine_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='positive-deviance-east-java')
print("Earth Engine OK")

In [ ]:
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2')
indo = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia'))
jatim = indo.filter(ee.Filter.eq('ADM1_NAME', 'Jawa Timur'))
print(jatim.size().getInfo())

In [ ]:
# 1) Daftar nama dari versi lama (39 unit), untuk melihat unit ekstranya
old = ee.FeatureCollection('FAO/GAUL/2015/level2') \
        .filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')) \
        .filter(ee.Filter.eq('ADM1_NAME', 'Jawa Timur'))
print(sorted(old.aggregate_array('ADM2_NAME').getInfo()))

# 2) Skema versi 2025 (nama kolom)
new = ee.FeatureCollection('FAO/GAUL/2025/level2')
print(new.first().propertyNames().getInfo())

In [ ]:
new = ee.FeatureCollection('FAO/GAUL/2025/level2') \
        .filter(ee.Filter.eq('GAUL0_NAME', 'Indonesia')) \
        .filter(ee.Filter.eq('GAUL1_NAME', 'Jawa Timur'))
print('jumlah 2025:', new.size().getInfo())

names_new = sorted(new.aggregate_array('GAUL2_NAME').getInfo())
names_old = set(old.aggregate_array('ADM2_NAME').getInfo())
print('hanya di 2025:', sorted(set(names_new) - names_old))
print('hanya di 2015:', sorted(names_old - set(names_new)))

# Luas Jember vs Kota Jember (versi 2015)
for n in ['Jember', 'Kota Jember']:
    g = old.filter(ee.Filter.eq('ADM2_NAME', n)).geometry()
    print(n, round(g.area(1).divide(1e6).getInfo(), 1), 'km2')

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/marifinilham-ds/east-java-positive-deviance-few/main/data/raw/bps/padi_jatim_2019_2022.csv"
padi = pd.read_csv(url)

bps = padi[['kabupaten_kota', 'jenis']].drop_duplicates().copy()
bps['gaul_name'] = bps.apply(
    lambda r: 'Kota ' + r.kabupaten_kota if r.jenis == 'kota' else r.kabupaten_kota,
    axis=1)

print('unit BPS:', len(bps))
print('di BPS, tidak ada di GAUL:', sorted(set(bps.gaul_name) - set(names_new)))
print('di GAUL, tidak ada di BPS:', sorted(set(names_new) - set(bps.gaul_name)))

In [ ]:
import pandas as pd

regions = new.select(['GAUL2_NAME'])
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
modis = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')

def to_df(fc):
    feats = fc.getInfo()['features']
    return pd.DataFrame([f['properties'] for f in feats])

frames = []
for y in [2019, 2020, 2021, 2022]:
    # Curah hujan: total tahunan dan musim kemarau (Jun-Sep), satuan mm
    rain_ann = chirps.filterDate(f'{y}-01-01', f'{y+1}-01-01').sum().rename('rain_annual_mm')
    rain_dry = chirps.filterDate(f'{y}-06-01', f'{y}-10-01').sum().rename('rain_dry_mm')
    rain_img = rain_ann.addBands(rain_dry)
    rain_fc = rain_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=5566)
    rain_df = to_df(rain_fc)

    # NDVI: rata-rata dan maksimum tahunan (skala asli 0.0001)
    nd = modis.filterDate(f'{y}-01-01', f'{y+1}-01-01')
    ndvi_img = (nd.mean().multiply(0.0001).rename('ndvi_mean')
                .addBands(nd.max().multiply(0.0001).rename('ndvi_max')))
    ndvi_fc = ndvi_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=250)
    ndvi_df = to_df(ndvi_fc)

    df = rain_df.merge(ndvi_df, on='GAUL2_NAME')
    df['tahun'] = y
    frames.append(df)
    print('selesai', y)

sat = pd.concat(frames, ignore_index=True).rename(columns={'GAUL2_NAME': 'gaul_name'})
print(sat.shape)
print(sat.isna().sum())
print(sat.groupby('tahun')[['rain_annual_mm', 'rain_dry_mm', 'ndvi_mean', 'ndvi_max']].describe().T.round(2))

In [ ]:
out = sat.merge(bps, on='gaul_name', how='left')
assert len(out) == 152 and out.kabupaten_kota.notna().all()

out = out[['kabupaten_kota', 'jenis', 'tahun',
           'rain_annual_mm', 'rain_dry_mm', 'ndvi_mean', 'ndvi_max']]
out[['rain_annual_mm', 'rain_dry_mm']] = out[['rain_annual_mm', 'rain_dry_mm']].round(2)
out[['ndvi_mean', 'ndvi_max']] = out[['ndvi_mean', 'ndvi_max']].round(4)

out.to_csv('iklim_ndvi_jatim_2019_2022.csv', index=False)
from google.colab import files
files.download('iklim_ndvi_jatim_2019_2022.csv')
print(out.head())

In [ ]:
import pandas as pd

viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG').select('avg_rad')
modis = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
lcover = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')

def to_df(fc):
    return pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])

# Porsi cropland per kabupaten (WorldCover 2021, kelas 40), tanpa reproject
wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
crop_img = wc.eq(40).rename('crop_share')
crop_df = (to_df(crop_img.reduceRegions(
    collection=regions, reducer=ee.Reducer.mean(), scale=100, tileScale=4))
    .rename(columns={'mean': 'crop_share'}))
print('crop_share selesai')

frames = []
for y in [2019, 2020, 2021, 2022]:
    # Cahaya malam VIIRS: rata-rata radiansi tahunan
    ntl = (viirs.filterDate(f'{y}-01-01', f'{y+1}-01-01')
           .mean().rename('ntl_mean'))
    ntl_df = (to_df(ntl.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=500, tileScale=4))
        .rename(columns={'mean': 'ntl_mean'}))

    # NDVI hanya di lahan pertanian (MODIS land cover: 12 = cropland, 14 = mosaik)
    lc = lcover.filterDate(f'{y}-01-01', f'{y+1}-01-01').first()
    crop_mask = lc.eq(12).Or(lc.eq(14))

    nd = modis.filterDate(f'{y}-01-01', f'{y+1}-01-01')
    ndvi_img = (nd.mean().multiply(0.0001).rename('ndvi_crop_mean')
                .addBands(nd.max().multiply(0.0001).rename('ndvi_crop_max'))
                .updateMask(crop_mask))
    ndvi_df = to_df(ndvi_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=250, tileScale=4))

    df = (ntl_df.merge(ndvi_df, on='GAUL2_NAME')
                .merge(crop_df, on='GAUL2_NAME'))
    df['tahun'] = y
    frames.append(df)
    print('selesai', y)

sat2 = (pd.concat(frames, ignore_index=True)
        .rename(columns={'GAUL2_NAME': 'gaul_name'}))

print(sat2.shape)
print(sat2.columns.tolist())
print(sat2.isna().sum())
print(sat2.groupby('tahun')[['ntl_mean', 'ndvi_crop_mean', 'ndvi_crop_max', 'crop_share']]
      .describe().T.round(3))

# Cek penamaan: kota besar harus terang dan porsi cropland-nya kecil
chk = (sat2[sat2.tahun == 2022].set_index('gaul_name')
       [['ntl_mean', 'crop_share', 'ndvi_crop_mean']])
print(chk.sort_values('ntl_mean', ascending=False).head(5).round(3))
print(chk.sort_values('crop_share').head(5).round(3))

In [ ]:
out = sat2.merge(bps, on='gaul_name', how='left')
assert len(out) == 152 and out.kabupaten_kota.notna().all()

out = out[['kabupaten_kota', 'jenis', 'tahun',
           'ntl_mean', 'crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']]
out[['ntl_mean']] = out[['ntl_mean']].round(3)
out[['crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']] = out[['crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']].round(4)

out.to_csv('viirs_cropndvi_jatim_2019_2022.csv', index=False)
from google.colab import files
files.download('viirs_cropndvi_jatim_2019_2022.csv')
print(out.head())